In [1]:
from pathlib import Path
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import colorcet as cc
import random
import pickle
import igraph as ig
import sys

sys.path.append("../../scripts")
import readwrite

cfg = readwrite.config()

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/norkin-organoid/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/norkin-organoid/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/norkin-organoid/lib/python3.11/site-packages/anndata/__ini

## Params

In [2]:
donor_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'donor_palette.csv',index_col=0)
gene_sets_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'gene_sets_palette.csv',index_col=0)
cell_type_palette = pd.read_csv(cfg['xenium_metadata_dir'] + 'cell_type_palette.csv',index_col=0)

gene_sets = (
    pd.Series({
        "Mitotic": ["MKI67", "CDK1", "UBE2C", "CENPF", "CD80", "ORC6", "STMN1", "TUBA1B"],
        "Goblet_TFF3": ["TFF3","STMN1","TUBA1B","CEACAM1","CEACAM6","RORC","DGKA","CEACAM8","NT5E","MIS18BP1","ATM","IL22RA1","NOTCH1","TUBB",],
        "Stress": ["AREG","CCL20","IL1A","CDKN2B","ANXA1","CXCL1","ISG15","CD68","ID1","IL1B",
                            "DGKA","CDKN1A","VSIR","CEACAM1","TFF3","CEACAM6","IRF1","CEACAM8","TUBB",],
        "Mesenchymal": ["FN1","CXCL6","C1S","CXCL5","C1R","NCAM1","CD74",],
        "Inflammatory": ["CXCL1","AREG","CDKN2B","CDKN1A","ANXA1"],
        "Goblet_MUC5AC": ["REG4", "MUC5AC", "DMBT1", "CCL28", "FOS", "IQGAP2", "JCHAIN", "IGKC", "TOX", "FAS", "IL1B", "RORC",],
        "Interferon": ["IDO1","MX1","IFIT3","CXCL11","CXCL10","ISG15","IFIT2","IRF1","STAT1","CXCL9",],
}).explode().reset_index())
gene_sets.columns = ["source", "target"]


xenium_dir = Path(cfg["xenium_processed_dir"])
xenium_count_correction_dir = Path(cfg["xenium_count_correction_dir"])
xenium_std_seurat_analysis_dir = Path(cfg["xenium_std_seurat_analysis_dir"])
xenium_cell_type_annotation_dir = Path(cfg["xenium_cell_type_annotation_dir"])
results_dir = Path(cfg["results_dir"])

# input params
cellcharter_dir = "cellcharter_cohort"
correction_method = "raw"
segmentation = "proseg_expected"
condition = ["CRC_PDO","CRC_PDO_CAF","CRC_PDO_DEV"]
panel = "all"
normalisation = "lognorm"
reference = "GEO_GSE178341"
method = "rctd_class_aware"
level = "Level1"
xenium_levels = ["segmentation", "condition", "panel", "donor", "sample"]
name_malignant = "Epi"

# qc params
min_n_counts = 20

# fixed params
BATCH_KEY = "dataset_id"
SPATIAL_KEY = "spatial"

segmentations_filter = [segmentation]
conditions_filter = condition
panels_filter = [panel] if panel != "all" else None

# Read results

In [3]:
# read samples
xenium_paths, xenium_annot_paths = readwrite.discover_xenium_paths(
    analysis_dir=xenium_std_seurat_analysis_dir,
    data_dir=xenium_dir,
    annotation_dir=xenium_cell_type_annotation_dir,
    correction_dir=xenium_count_correction_dir,
    normalisation=normalisation,
    reference=reference,
    method=method,
    level=level,
    correction_methods_filter=[correction_method],
    segmentations_filter=[segmentation],
    conditions_filter=conditions_filter,
    panels_filter=panels_filter,
)


# set transcripts=True to load individual transcripts positions)
sds = {}
sds["raw"] = readwrite.read_xenium_samples(
    xenium_paths["raw"], anndata=False, 
    cells_boundaries=True,
    pool_mode="thread", max_workers=6)

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/norkin-organoid/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/norkin-organoid/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/norkin-organoid/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/norkin-organoid/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/workflow/analysis/xenium/../../scripts/readwri

## segment organoids

In [ ]:
# Distance threshold
r = 0

# load manual refinement
dict_manual_refinement = pickle.load(open(cfg['xenium_metadata_dir'] + "dict_manual_refinement.pkl", "rb"))

for k, sd in sds["raw"].items():
    print(k)
    k_str = "_".join(k)  

    gdf = sd['cells_boundaries']
    dict_manual_refinement[k]

    rows, cols = [], []
    for i, geom in enumerate(gdf.geometry):
        for j in gdf.sindex.query(geom.buffer(r)):
            if i < j and geom.distance(gdf.geometry[j]) <= r:
                rows.append(i)
                cols.append(j)

    # Build adjacency matrix
    n = len(gdf)
    A = csr_matrix(( [1] * len(rows), (rows, cols) ), shape=(n, n))

    # Symmetrize (undirected graph)
    A = A + A.T

    # Connected components
    n_components, component_labels = connected_components(A, directed=False)

    # Cluster labels to  furhter split connected components.
    random.seed(0)
    g = ig.Graph(n=n, edges=zip(rows,cols), directed=False)
    partition = g.community_leiden(objective_function='modularity')
    cluster_labels = np.array(partition.membership,dtype=np.int32)

    # Store results
    gdf["component_labels"] = component_labels
    gdf["cluster_labels"] = cluster_labels

    idx_manual = gdf['component_labels'].isin(dict_manual_refinement[k])

    gdf["component_labels"] = gdf["component_labels"].apply(lambda x: f"{k_str}_comp{x}")
    gdf["cluster_labels"]   = gdf["cluster_labels"].apply(lambda x: f"{k_str}_clus{x}")

    # add manual refinement
    gdf["component_and_cluster_labels"] = gdf["component_labels"]
    gdf.loc[idx_manual,"component_and_cluster_labels"] = gdf.loc[idx_manual,"cluster_labels"]

    # Plot all polygons, colored by cluster_id
    # fig, ax = plt.subplots(figsize=(10, 10))
    # gdf.plot(column="component_labels", cmap=cc.cm.glasbey, legend=True, ax=ax,alpha=.5,facecolor='none',aspect=1)
    # ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
    # plt.show()

    # fig, ax = plt.subplots(figsize=(10, 10))
    # gdf.plot(column="component_and_cluster_labels", cmap=cc.cm.glasbey, legend=True, ax=ax,alpha=.5,facecolor='none',aspect=1)
    # ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
    # plt.show()

    # fig, ax = plt.subplots(figsize=(10, 10))
    # gdf.plot(column="cluster_labels", cmap=cc.cm.glasbey, legend=True, ax=ax,alpha=.5,facecolor='none',aspect=1)
    # ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
    # plt.show()

    # for i in range(gdf['component_labels'].nunique()):
    #     gdf_ = gdf[gdf['component_labels'] == i]
    #     if len(gdf_) < 40 or all(gdf_['cluster_labels'].value_counts()<20) or gdf_['cluster_labels'].nunique()==1:
    #         continue

    #     fig, ax = plt.subplots(1,2,figsize=(5, 5))
    #     fig.suptitle(i,y=.9)
    #     gdf[gdf['component_labels'] == i].plot(column="component_labels", cmap=cc.cm.glasbey, legend=False,alpha=.5,facecolor='none',aspect=1,ax=ax[0])

    #     gdf[gdf['component_labels'] == i].plot(column="cluster_labels", cmap=cc.cm.glasbey, legend=True,alpha=.5,facecolor='none',aspect=1,ax=ax[1])
        
    #     plt.show()


('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '7_OY6Hsmall', 'output-XETG00059__0033028__7_OY6Hsmall__20250811__161841')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '8_OY6Hsmallmiddle', 'output-XETG00059__0033028__8_OY6Hsmallmiddle__20250811__161841')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '12_OY6Hbighuge', 'output-XETG00059__0033028__12_OY6Hbighuge__20250811__161841')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '10_OY6Hmiddlebig', 'output-XETG00059__0033028__10_OY6Hmiddlebig__20250811__161841')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '1BI7', 'output-XETG00059__0003881__1BI7__20250505__170804')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '1HVQ', 'output-XETG00059__0003381__1HVQ__20250505__170803')
('proseg_expected', 'CRC_PDO_DEV', 'hImmune_v1_mm', '9_11_OY6H_middle_and_big', 'output-XETG00059__0033028__9_11_OY6H_middle_and_big__20250811__161841')
('proseg_expected', 'CRC_PDO_CAF', 'hImmune_v1_dapi', '8samples', 'output-XETG00059__0033

('proseg_expected', 'CRC_PDO', 'hImmune_v1_mm', 'OWMY', 'output-XETG00059__0021741__OWMY__20250319__172035')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_mm', 'OYRI', 'output-XETG00059__0021741__OYRI__20250319__172035')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_mm', 'OUC1', 'output-XETG00059__0021738__OUC1__20250319__172035')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_mm', 'O056', 'output-XETG00059__0021741__O056__20250319__172035')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_mm', 'OAFN', 'output-XETG00059__0021738__OAFN__20250319__172035')
('proseg_expected', 'CRC_PDO', 'hImmune_v1_dapi', '18samples', 'output-XETG00059__0033132__PDO_18samples__20250821__124603')


## write results

In [ ]:
gdf_all = pd.concat({k:sd['cells_boundaries'] for k, sd in sds["raw"].items()})
gdf_all = gdf_all.reset_index()
gdf_all = gdf_all.rename(columns=dict(zip([f"level_{i}" for i in range(5)],xenium_levels)))
gdf_all["cell_id"] = "proseg-"+gdf_all["cell_id"].astype(str)
gdf_all['full_id'] = gdf_all[xenium_levels+['cell_id']].apply(lambda x: "_".join(x),axis=1)
p=Path(cfg['results_dir'] + "xenium/segment_organoids/organoids_ids.parquet")
p.parent.mkdir(parents=True, exist_ok=True)
gdf_all = gdf_all.set_index('full_id')
gdf_all.to_parquet(p)

## read results

In [14]:
import colorcet as cc
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import sys
from pathlib import Path

sys.path.append("../../scripts")
import readwrite

cfg = readwrite.config()

p=Path(cfg['results_dir'] + "xenium/segment_organoids/organoids_ids.parquet")

gdf = gpd.read_parquet(p)

# connected components (baseline)
gdf["component_labels"]
# leiden clusters (not recommended)
gdf["cluster_labels"]
# connected components slightly sub-divided with leiden clusters (recommended)
gdf["component_and_cluster_labels"]

plt.style.use('dark_background')
# plot random sample
for sample in gdf['sample'].unique():
    print(sample)
    gdf_ = gdf[gdf['sample'] == sample]

    fig, ax = plt.subplots(figsize=(20, 20))
    gdf_.plot(column="component_and_cluster_labels", cmap=cc.cm.glasbey_light, legend=False, ax=ax,alpha=1,facecolor='none',aspect=1)
    # ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
    p = Path(cfg['figures_dir']+f"xenium/segment_organoids/segment_organoids_sample_{sample}.png")
    p.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(p,bbox_inches='tight')
    plt.close()

output-XETG00059__0033028__7_OY6Hsmall__20250811__161841
output-XETG00059__0033028__8_OY6Hsmallmiddle__20250811__161841
output-XETG00059__0033028__12_OY6Hbighuge__20250811__161841
output-XETG00059__0033028__10_OY6Hmiddlebig__20250811__161841
output-XETG00059__0003881__1BI7__20250505__170804
output-XETG00059__0003381__1HVQ__20250505__170803
output-XETG00059__0033028__9_11_OY6H_middle_and_big__20250811__161841
output-XETG00059__0033131__PDO_8samples__20250821__124602
output-XETG00059__0003381__1CNN__20250505__170803
output-XETG00059__0003381__1J25__20250505__170803
output-XETG00059__0003381__OWJ3__20250505__170803
output-XETG00059__0003881__1GVB__20250505__170804
output-XETG00059__0003881__1GNS__20250505__170804
output-XETG00059__0003881__12NM__20250505__170804
output-XETG00059__0003881__1FMS__20250505__170803
output-XETG00059__0003381__131N__20250505__170803
output-XETG00059__0003881__1CI5__20250505__170804
output-XETG00059__0003881__169V_not_1JSK_20250505__170804
output-XETG00059__0003

## manual refinement of assignments

In [ ]:
keys = list(sds["raw"].keys())
dict_manual_refinement = {k:[] for k in keys}
iter_ = 0

In [ ]:
# Distance threshold
r = 0

k = keys[iter_]
sd = sds["raw"][k]
iter_+=1

# if k[3] == '8_OY6Hsmallmiddle':

print(k)

gdf = sd['cells_boundaries']
dict_manual_refinement[k]

rows, cols = [], []
for i, geom in enumerate(gdf.geometry):
    for j in gdf.sindex.query(geom.buffer(r)):
        if i < j and geom.distance(gdf.geometry[j]) <= r:
            rows.append(i)
            cols.append(j)

# Build adjacency matrix
n = len(gdf)
A = csr_matrix(( [1] * len(rows), (rows, cols) ), shape=(n, n))

# Symmetrize (undirected graph)
A = A + A.T

# Connected components
n_components, component_labels = connected_components(A, directed=False)

# Extract the cluster labels for each cell.
random.seed(0)
g = ig.Graph(n=n, edges=zip(rows,cols), directed=False)
partition = g.community_leiden(objective_function='modularity')
cluster_labels = np.array(partition.membership,dtype=np.int32)

gdf["component_labels"] = component_labels
gdf["cluster_labels"] = cluster_labels
gdf["component_and_cluster_labels"] = component_labels

idx_manual = gdf['component_labels'].isin(dict_manual_refinement[k])
gdf.loc[idx_manual,"component_and_cluster_labels"] = gdf.loc[idx_manual,"cluster_labels"]



# Plot all polygons, colored by cluster_id
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column="component_labels", cmap=cc.cm.glasbey, legend=True, ax=ax,alpha=.5,facecolor='none',aspect=1)
ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
plt.show()

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column="component_and_cluster_labels", cmap=cc.cm.glasbey, legend=True, ax=ax,alpha=.5,facecolor='none',aspect=1)
ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
plt.show()

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column="cluster_labels", cmap=cc.cm.glasbey, legend=True, ax=ax,alpha=.5,facecolor='none',aspect=1)
ax.set_title("Polygon clusters based on distance threshold r={}".format(r))
plt.show()

for i in range(gdf['component_labels'].nunique()):
    gdf_ = gdf[gdf['component_labels'] == i]
    if len(gdf_) < 40 or all(gdf_['cluster_labels'].value_counts()<20) or gdf_['cluster_labels'].nunique()==1:
        continue

    fig, ax = plt.subplots(1,2,figsize=(5, 5))
    fig.suptitle(i,y=.9)
    gdf[gdf['component_labels'] == i].plot(column="component_labels", cmap=cc.cm.glasbey, legend=False,alpha=.5,facecolor='none',aspect=1,ax=ax[0])

    gdf[gdf['component_labels'] == i].plot(column="cluster_labels", cmap=cc.cm.glasbey, legend=True,alpha=.5,facecolor='none',aspect=1,ax=ax[1])
    
    plt.show()


In [ ]:
# with open(cfg['xenium_metadata_dir'] + "dict_manual_refinement.pkl", "wb") as f:
#     pickle.dump(dict_manual_refinement, f)